In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.pipeline import Pipeline
from pathlib import Path
import sys
base_dir = Path().resolve().parent
sys.path.append(str(base_dir / "src"))
from helpers.load_data import load_data
import warnings
warnings.filterwarnings("ignore")

In [2]:
dataset = "telco_customer_churn_clean.csv"
X_train, X_test, y_train, y_test = load_data(dataset)

Dataset 'telco_customer_churn_clean.csv' loaded successfully.

TotalCharges was typecasted to numerical.

Null values were removed.


Dataset size : 
7032 rows
21 columns


Successfully dropped the columns
   -customerID
   -gender
   -PhoneService
   -TotalCharges


Dataset split complete.


In [4]:
model = LogisticRegression()

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_cols = ["SeniorCitizen", "Partner", "Dependents",
           "MultipleLines", "InternetService_DSL", "InternetService_Fiber optic",
           "OnlineSecurity", "OnlineBackup", "DeviceProtection",
           "TechSupport", "StreamingTV", "StreamingMovies",
           "Contract_Month-to-month", "Contract_One year",
           "PaperlessBilling", "PaymentMethod_Bank transfer (automatic)",
           "PaymentMethod_Credit card (automatic)",
           "PaymentMethod_Electronic check"]

smotenc = SMOTENC(
    categorical_features=cat_cols,
    random_state=42
)

pipeline = Pipeline([
    ("smote",smotenc),
    ("model",model)
])

metrics = ["accuracy", "precision", "recall", "f1"]

scores_model = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=skf,
    scoring=metrics
)
temp_arr = np.zeros((6,1))
temp_df = pd.DataFrame(temp_arr,index=scores_model.keys(),columns=["Mean_Score"])
for metric in metrics:
    mean_score = scores_model[f"test_{metric}"].mean()
    if f"test_{metric}" in scores_model.keys():
        temp_df.loc[f"test_{metric}","Mean_Score"] = mean_score
    print(f"Metric : {metric} = Mean Score : {mean_score}")
temp_df.to_csv(base_dir / "data" / "smotenc_log_reg_NoRegu.csv")

Metric : accuracy = Mean Score : 0.7557333333333334
Metric : precision = Mean Score : 0.5274839005813188
Metric : recall = Mean Score : 0.7752508361204014
Metric : f1 = Mean Score : 0.6277257287744683


In [5]:
c = [0.01, 0.03, 0.09, 0.3, 0.9, 3]

scores = {}

for c_val in c:
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    model = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=c_val,
    )
    smotenc = SMOTENC(
        categorical_features=cat_cols,
        random_state=42
    )
    pipeline = Pipeline([
        ("smote",smotenc),
        ("model",model)
    ])
    scores_model = cross_validate(
        estimator=pipeline,
        scoring=metrics,
        cv=skf,
        X=X_train,
        y=y_train
    )
    scores[c_val]=scores_model

new_arr = np.zeros((len(metrics),len(c)))
new_df = pd.DataFrame(new_arr,index=metrics,columns=c)

for c_val in c:
    curr_score = scores[c_val]
    print(f"For C value of {c_val}")
    for metric in metrics:
        mean_score = curr_score[f"test_{metric}"].mean()
        new_df.loc[metric,c_val]=mean_score
        print(f"Metric : {metric} = Mean_score : {mean_score}")
    print()

new_df.to_csv(base_dir / "data" / "smotenc_lin_reg_l2_regu_DiffC.csv")

For C value of 0.01
Metric : accuracy = Mean_score : 0.7502222222222221
Metric : precision = Mean_score : 0.5204764097010162
Metric : recall = Mean_score : 0.7725752508361204
Metric : f1 = Mean_score : 0.6217704269700499

For C value of 0.03
Metric : accuracy = Mean_score : 0.7562666666666666
Metric : precision = Mean_score : 0.5285363582711168
Metric : recall = Mean_score : 0.7678929765886288
Metric : f1 = Mean_score : 0.6260645928013843

For C value of 0.09
Metric : accuracy = Mean_score : 0.7564444444444444
Metric : precision = Mean_score : 0.5289071088255742
Metric : recall = Mean_score : 0.7678929765886289
Metric : f1 = Mean_score : 0.6262731105739248

For C value of 0.3
Metric : accuracy = Mean_score : 0.7555555555555556
Metric : precision = Mean_score : 0.5275560808192804
Metric : recall = Mean_score : 0.7705685618729097
Metric : f1 = Mean_score : 0.6261995223822436

For C value of 0.9
Metric : accuracy = Mean_score : 0.7553777777777777
Metric : precision = Mean_score : 0.527071

In [13]:
model = LogisticRegression()

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=45
)

smoteenn = SMOTEENN(random_state=42)

pipeline = Pipeline([
    ("smote",smoteenn),
    ("model",model)
])

model_scores = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=skf,
    scoring=metrics
)

new_arr = np.zeros((4,1))
new_df = pd.DataFrame(new_arr, index=metrics, columns=["Mean_Score"])
for metric in metrics:
    mean_score = model_scores[f"test_{metric}"].mean()
    new_df.loc[f"test_{metric}","Mean_Score"] = mean_score
    print(f"Metric : {metric} = Mean_Score : {mean_score}")

new_df.to_csv(base_dir / "data" / "smoteenn_log_reg_NoRegu.csv")

Metric : accuracy = Mean_Score : 0.7118222222222222
Metric : precision = Mean_Score : 0.47698787277402854
Metric : recall = Mean_Score : 0.8595317725752508
Metric : f1 = Mean_Score : 0.6129997347781349


In [16]:
c = [0.001, 0.003, 0.01, 0.03, 0.09, 0.3, 0.9, 3]

scores = {}

for c_val in c:
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    model = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=c_val,
    )
    smoteenn = SMOTEENN(
        random_state=42
    )
    pipeline = Pipeline([
        ("smote",smoteenn),
        ("model",model)
    ])
    scores_model = cross_validate(
        estimator=pipeline,
        scoring=metrics,
        cv=skf,
        X=X_train,
        y=y_train
    )
    scores[c_val]=scores_model

new_arr = np.zeros((len(metrics),len(c)))
new_df = pd.DataFrame(new_arr,index=metrics,columns=c)

for c_val in c:
    curr_score = scores[c_val]
    print(f"For C value of {c_val}")
    for metric in metrics:
        mean_score = curr_score[f"test_{metric}"].mean()
        new_df.loc[metric,c_val]=mean_score
        print(f"Metric : {metric} = Mean_score : {mean_score}")
    print()

new_df.to_csv(base_dir / "data" / "smoteenn_lin_reg_l2_regu_DiffC.csv")

For C value of 0.001
Metric : accuracy = Mean_score : 0.6568888888888889
Metric : precision = Mean_score : 0.4307243857804792
Metric : recall = Mean_score : 0.9043478260869564
Metric : f1 = Mean_score : 0.5835147538873028

For C value of 0.003
Metric : accuracy = Mean_score : 0.6775111111111112
Metric : precision = Mean_score : 0.4462642702998104
Metric : recall = Mean_score : 0.8856187290969899
Metric : f1 = Mean_score : 0.593461604973743

For C value of 0.009
Metric : accuracy = Mean_score : 0.6892444444444445
Metric : precision = Mean_score : 0.4558734914692428
Metric : recall = Mean_score : 0.8735785953177256
Metric : f1 = Mean_score : 0.5990593891521047

For C value of 0.01
Metric : accuracy = Mean_score : 0.6901333333333332
Metric : precision = Mean_score : 0.45659608038966004
Metric : recall = Mean_score : 0.8722408026755852
Metric : f1 = Mean_score : 0.5993705975256097

For C value of 0.03
Metric : accuracy = Mean_score : 0.6984888888888888
Metric : precision = Mean_score : 0.4

In [20]:
model = LogisticRegression()

sfk = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

smotetomek = SMOTETomek(random_state=42)

pipeline = Pipeline([
    ("smote",smotetomek),
    ("model",model)
])

model_scores = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=sfk,
    scoring=metrics
)

new_arr = np.zeros((4,1))
new_df = pd.DataFrame(new_arr,index=metrics,columns=["Mean_Score"])

for metric in metrics:
    mean_score = model_scores[f"test_{metric}"].mean()
    new_df.loc[metric,"Mean_Score"] = mean_score
    print(f"Metric : {metric} = Mean Score : {mean_score}")

new_df.to_csv(base_dir / "data" / "smotetomek_lin_reg_NoRegu.csv")

Metric : accuracy = Mean Score : 0.7509333333333333
Metric : precision = Mean Score : 0.520915498582823
Metric : recall = Mean Score : 0.785953177257525
Metric : f1 = Mean Score : 0.6264812018328881


In [22]:
c = [0.001, 0.003, 0.01, 0.03, 0.09, 0.3, 0.9, 3]

scores = {}

for c_val in c:
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    model = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=c_val
    )
    smotetomek = SMOTETomek(
        random_state=42
    )
    pipeline = Pipeline([
        ("smote",smotetomek),
        ("model",model)
    ])
    scores_model = cross_validate(
        estimator=pipeline,
        scoring=metrics,
        cv=skf,
        X=X_train,
        y=y_train
    )
    scores[c_val]=scores_model

new_arr = np.zeros((len(metrics),len(c)))
new_df = pd.DataFrame(new_arr,index=metrics,columns=c)

for c_val in c:
    curr_score = scores[c_val]
    print(f"For C value of {c_val}")
    for metric in metrics:
        mean_score = curr_score[f"test_{metric}"].mean()
        new_df.loc[metric,c_val]=mean_score
        print(f"Metric : {metric} = Mean_score : {mean_score}")
    print()

new_df.to_csv(base_dir / "data" / "smotetomek_lin_reg_l2_regu_DiffC.csv")

For C value of 0.001
Metric : accuracy = Mean_score : 0.7233777777777778
Metric : precision = Mean_score : 0.48815250066950283
Metric : recall = Mean_score : 0.8341137123745819
Metric : f1 = Mean_score : 0.615840908740388

For C value of 0.003
Metric : accuracy = Mean_score : 0.7336888888888888
Metric : precision = Mean_score : 0.4996532632980711
Metric : recall = Mean_score : 0.8107023411371237
Metric : f1 = Mean_score : 0.6181527133231212

For C value of 0.01
Metric : accuracy = Mean_score : 0.7445333333333333
Metric : precision = Mean_score : 0.5128449209924828
Metric : recall = Mean_score : 0.7892976588628763
Metric : f1 = Mean_score : 0.621535328804268

For C value of 0.03
Metric : accuracy = Mean_score : 0.7507555555555555
Metric : precision = Mean_score : 0.5207478887965172
Metric : recall = Mean_score : 0.779933110367893
Metric : f1 = Mean_score : 0.624416833646986

For C value of 0.09
Metric : accuracy = Mean_score : 0.7518222222222222
Metric : precision = Mean_score : 0.52206